# Req 3: Compelling Warehouse Queries — "Predict the Future"

Same 3 queries as Part1_Group11.sql Req 3, now with actual data loaded (4 days of ETL).
Each query JOINs FactSales to all 6 Dims.

In [1]:
import pyodbc
import pandas as pd

conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=localhost;"
    r"DATABASE=WWI_DM;"
    r"Trusted_Connection=yes;"
)

def sql(query):
    return pd.read_sql(query, conn)

print("Connected to WWI_DM")

Connected to WWI_DM


## Query 1: Supplier Performance by City

**Business scenario:** The company wants to optimize supplier contracts by region. If Supplier A's products sell 3x more in a city than Supplier B's, we should increase Supplier A's stock in that region.

**Predict the Future:** If Supplier X dominates City Y → negotiate better terms. If a supplier is weak in a region → consider replacing.

In [4]:
# Same scenario as Part1_Group11.sql Req 3 Query 1, enhanced with ranking + revenue share
sql("""
    WITH CitySupplier AS (
        SELECT
            l.CityName, l.StateProvCode,
            s.FullName AS SupplierName, s.SupplierCategoryName,
            COUNT(DISTINCT p.ProductKey) AS Products,
            SUM(f.Quantity) AS TotalQuantity,
            SUM(f.TotalAfterTax) AS TotalRevenue
        FROM dbo.FactSales f
        JOIN dbo.DimLocation l     ON f.LocationKey    = l.LocationKey
        JOIN dbo.DimSuppliers s    ON f.SupplierKey    = s.SupplierKey
        JOIN dbo.DimCustomers c    ON f.CustomerKey    = c.CustomerKey
        JOIN dbo.DimProducts p     ON f.ProductKey     = p.ProductKey
        JOIN dbo.DimSalesPeople sp ON f.SalespersonKey = sp.SalespersonKey
        JOIN dbo.DimDate d         ON f.DateKey        = d.DateKey
        GROUP BY l.CityName, l.StateProvCode, s.FullName, s.SupplierCategoryName
    ),
    CityTotal AS (
        SELECT CityName, SUM(TotalRevenue) AS CityRevenue FROM CitySupplier GROUP BY CityName
    )
    SELECT TOP 30
        cs.CityName, cs.StateProvCode,
        cs.SupplierName, cs.SupplierCategoryName,
        cs.Products, cs.TotalQuantity, cs.TotalRevenue,
        CAST(cs.TotalRevenue * 100.0 / ct.CityRevenue AS DECIMAL(5,1)) AS RevenueSharePct,
        RANK() OVER (PARTITION BY cs.CityName ORDER BY cs.TotalRevenue DESC) AS SupplierRankInCity
    FROM CitySupplier cs
    JOIN CityTotal ct ON cs.CityName = ct.CityName
    ORDER BY ct.CityRevenue DESC, cs.TotalRevenue DESC
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_99884\4066123149.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,CityName,StateProvCode,SupplierName,SupplierCategoryName,Products,TotalQuantity,TotalRevenue,RevenueSharePct,SupplierRankInCity
0,Jones Creek,TX,"Litware, Inc.",Packaging Supplier,5,420,10934.89,71.2,1
1,Jones Creek,TX,"Fabrikam, Inc.",Clothing Supplier,2,204,4222.80,27.5,2
2,Jones Creek,TX,Graphic Design Institute,Novelty Goods Supplier,1,7,104.65,0.7,3
3,Jones Creek,TX,The Phone Company,Novelty Goods Supplier,1,3,86.25,0.6,4
4,Lynne,FL,"Litware, Inc.",Packaging Supplier,2,324,11337.39,74.4,1
5,Lynne,FL,"Fabrikam, Inc.",Clothing Supplier,4,162,3450.00,22.6,2
6,Lynne,FL,Northwind Electric Cars,Toy Supplier,1,10,287.50,1.9,3
7,Lynne,FL,Graphic Design Institute,Novelty Goods Supplier,2,11,164.45,1.1,4
8,Marcell,MN,"Litware, Inc.",Packaging Supplier,2,170,12696.00,96.6,1
9,Marcell,MN,"Fabrikam, Inc.",Clothing Supplier,2,12,453.10,3.4,2


**Conclusion:**

Litware, Inc. (Packaging Supplier) ranks #1 in every city, with revenue share ranging from 56.9% (New Zion) to 99.7% (Airport Drive). This is far more than 3x Supplier B — in most cities Litware has 5-10x the revenue of the next supplier.

**Action per the scenario:**
- Increase Litware stock across all regions — they are the dominant revenue driver
- However, this extreme concentration is also a risk — if Litware has supply issues, most cities lose 70-99% of revenue
- Fabrikam (Clothing) is consistently #2 but with a huge gap — explore expanding their product range to reduce Litware dependency
- New Zion (SC) is the most diversified city (4 suppliers, #1 at 56.9%) — this is the healthiest distribution

---

## Query 2: Salesperson × Supplier Performance

**Business scenario:** Which salesperson sells which supplier's products best? By matching salespeople to the suppliers they're most effective with, we can optimize assignments and improve revenue.

**Predict the Future:**
- Salesperson X generates 80% of their revenue from Supplier A's products → assign more of that supplier's accounts to them
- Salesperson Y is weak with a key supplier → provide product training or reassign
- New supplier onboarded? → assign the salesperson who performs best with similar supplier categories

In [2]:
sql("""
    WITH SalespersonSupplier AS (
        SELECT
            sp.FullName AS SalespersonName,
            s.FullName AS SupplierName, s.SupplierCategoryName,
            COUNT(DISTINCT p.ProductKey) AS Products,
            SUM(f.Quantity) AS TotalQuantity,
            SUM(f.TotalAfterTax) AS TotalRevenue
        FROM dbo.FactSales f
        JOIN dbo.DimSalesPeople sp ON f.SalespersonKey = sp.SalespersonKey
        JOIN dbo.DimSuppliers s    ON f.SupplierKey    = s.SupplierKey
        JOIN dbo.DimProducts p     ON f.ProductKey     = p.ProductKey
        JOIN dbo.DimCustomers c    ON f.CustomerKey    = c.CustomerKey
        JOIN dbo.DimLocation l     ON f.LocationKey    = l.LocationKey
        JOIN dbo.DimDate d         ON f.DateKey        = d.DateKey
        GROUP BY sp.FullName, s.FullName, s.SupplierCategoryName
    ),
    SalespersonTotal AS (
        SELECT SalespersonName, SUM(TotalRevenue) AS SalespersonRevenue
        FROM SalespersonSupplier GROUP BY SalespersonName
    )
    SELECT TOP 30
        ss.SalespersonName, ss.SupplierName, ss.SupplierCategoryName,
        ss.Products, ss.TotalQuantity, ss.TotalRevenue,
        CAST(ss.TotalRevenue * 100.0 / st.SalespersonRevenue AS DECIMAL(5,1)) AS RevenueSharePct,
        RANK() OVER (PARTITION BY ss.SalespersonName ORDER BY ss.TotalRevenue DESC) AS SupplierRankPerSalesperson
    FROM SalespersonSupplier ss
    JOIN SalespersonTotal st ON ss.SalespersonName = st.SalespersonName
    ORDER BY st.SalespersonRevenue DESC, ss.TotalRevenue DESC
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_99884\4066123149.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,SalespersonName,SupplierName,SupplierCategoryName,Products,TotalQuantity,TotalRevenue,RevenueSharePct,SupplierRankPerSalesperson
0,Amy Trefl,"Litware, Inc.",Packaging Supplier,17,2299,45732.92,61.8,1
1,Amy Trefl,"Fabrikam, Inc.",Clothing Supplier,22,1063,22885.00,30.9,2
2,Amy Trefl,Northwind Electric Cars,Toy Supplier,5,29,2990.00,4.0,3
3,Amy Trefl,Graphic Design Institute,Novelty Goods Supplier,9,76,1136.20,1.5,4
4,Amy Trefl,The Phone Company,Novelty Goods Supplier,4,31,1092.50,1.5,5
5,Amy Trefl,"Contoso, Ltd.",Novelty Goods Supplier,1,7,128.80,0.2,6
6,Anthony Grosse,"Litware, Inc.",Packaging Supplier,23,2903,42455.14,63.3,1
7,Anthony Grosse,"Fabrikam, Inc.",Clothing Supplier,26,853,18089.50,27.0,2
8,Anthony Grosse,Northwind Electric Cars,Toy Supplier,4,30,2932.50,4.4,3
9,Anthony Grosse,Graphic Design Institute,Novelty Goods Supplier,18,135,2018.25,3.0,4


**Conclusion:**

Litware (Packaging) is #1 for almost every salesperson at 59-63% revenue share — except **Hudson Hollinworth**, where Fabrikam (Clothing) leads at 52% vs Litware at 32%.

**Action per the scenario:**
- Hudson is strong with Clothing products → assign more Fabrikam/Clothing accounts to Hudson
- Amy, Anthony, Jack are Litware-dominant → deploy them in regions where Packaging push is needed
- Lily Code has 15% in Northwind (Toy Supplier), much higher than others (4-6%) → potential Toy category specialist
- Overall Litware dependency mirrors Query 1 findings → need salesperson training/incentives to grow Fabrikam and Northwind revenue

---

## Query 3: Salesperson Efficiency by Customer Category

**Business scenario:** Not all salespeople perform equally across customer types. A salesperson who excels with Corporate clients might struggle with Novelty Shops. We want to find the best match between salesperson and customer segment.

**Predict the Future:**
- Salesperson X has highest revenue/customer for "Corporate" → assign more Corporate accounts
- Salesperson Y is inefficient with "Novelty Shop" → reassign or provide training
- New customer segment growing? → assign the salesperson who performs best in similar segments

In [5]:
sql("""
    WITH SalespersonCategory AS (
        SELECT
            sp.FullName AS SalespersonName,
            c.CustomerCategoryName,
            COUNT(DISTINCT c.CustomerKey) AS UniqueCustomers,
            SUM(f.Quantity) AS TotalQuantity,
            SUM(f.TotalAfterTax) AS TotalRevenue,
            SUM(f.TotalAfterTax) / COUNT(DISTINCT c.CustomerKey) AS RevenuePerCustomer
        FROM dbo.FactSales f
        JOIN dbo.DimSalesPeople sp ON f.SalespersonKey = sp.SalespersonKey
        JOIN dbo.DimCustomers c    ON f.CustomerKey    = c.CustomerKey
        JOIN dbo.DimProducts p     ON f.ProductKey     = p.ProductKey
        JOIN dbo.DimLocation l     ON f.LocationKey    = l.LocationKey
        JOIN dbo.DimSuppliers s    ON f.SupplierKey    = s.SupplierKey
        JOIN dbo.DimDate d         ON f.DateKey        = d.DateKey
        GROUP BY sp.FullName, c.CustomerCategoryName
    ),
    CategoryTotal AS (
        SELECT CustomerCategoryName, SUM(TotalRevenue) AS CategoryRevenue
        FROM SalespersonCategory GROUP BY CustomerCategoryName
    )
    SELECT TOP 30
        sc.CustomerCategoryName, sc.SalespersonName,
        sc.UniqueCustomers, sc.TotalQuantity, sc.TotalRevenue,
        sc.RevenuePerCustomer,
        CAST(sc.TotalRevenue * 100.0 / ct.CategoryRevenue AS DECIMAL(5,1)) AS RevenueSharePct,
        RANK() OVER (PARTITION BY sc.CustomerCategoryName ORDER BY sc.TotalRevenue DESC) AS SalespersonRankInCategory
    FROM SalespersonCategory sc
    JOIN CategoryTotal ct ON sc.CustomerCategoryName = ct.CustomerCategoryName
    ORDER BY ct.CategoryRevenue DESC, sc.TotalRevenue DESC
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_99884\4066123149.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,CustomerCategoryName,SalespersonName,UniqueCustomers,TotalQuantity,TotalRevenue,RevenuePerCustomer,RevenueSharePct,SalespersonRankInCategory
0,Novelty Shop,Anthony Grosse,17,3152,51558.54,3032.855294,14.6,1
1,Novelty Shop,Hudson Hollinworth,9,1536,40172.26,4463.584444,11.3,2
2,Novelty Shop,Jack Potter,16,2327,40141.10,2508.818750,11.3,3
3,Novelty Shop,Sophia Hinton,15,1585,37976.97,2531.798000,10.7,4
4,Novelty Shop,Taj Shand,20,2084,37546.13,1877.306500,10.6,5
5,Novelty Shop,Amy Trefl,12,1729,37238.15,3103.179166,10.5,6
6,Novelty Shop,Hudson Onslow,8,1082,31626.50,3953.312500,8.9,7
7,Novelty Shop,Kayla Woodcock,10,1637,27253.51,2725.351000,7.7,8
8,Novelty Shop,Archer Lamble,12,1468,25456.17,2121.347500,7.2,9
9,Novelty Shop,Lily Code,11,1076,25134.17,2284.924545,7.1,10


**Conclusion:**

Novelty Shop has the largest total revenue but salesperson share is evenly spread (7-14%) — low dependency risk. Gift Store has Lily Code leading at 27%. Supermarket has Amy Trefl at 28%. Kayla Woodcock has 1 Supermarket customer generating $9,085 — highest RevenuePerCustomer overall.

**Action per the scenario:**
- Lily Code → assign more Gift Store customers (category leader at 27%)
- Amy Trefl → maintain/expand Supermarket assignment (leader at 28%)
- Kayla Woodcock → strong in both Corporate and Supermarket, candidate for high-value customer specialist
- Novelty Shop → well-distributed across salespeople, keep current assignments
- Anthony Grosse → Corporate 3rd (17%) and Novelty Shop 1st (15%) → can handle both categories